In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
subjects_prompt = """Generate a comma separated list of between 2 and 5 examples releated to the topic: {topic}"""

In [4]:
from pydantic import BaseModel
from typing import TypedDict, Annotated
import operator

class Subjects(BaseModel):
    subjects: list[str]

In [14]:
class OverallState(TypedDict):
    topic: str
    subjects: list
    jokes: Annotated[list, operator.add]
    best_selected_joke: str
    
class jokeState(TypedDict):
    subject: str


In [21]:
def generate_topics(state: OverallState):
    
    thread_name = threading.current_thread().name
    print(f"Thread name: {thread_name}.... inside generate_topics")
    
    prompt = subjects_prompt.format(topic=state["topic"])
    response = model.with_structured_output(Subjects).invoke(prompt)
    print(response.subjects)
    
    return {"subjects":response.subjects}

In [22]:
generate_topics({"topic":"Animal"})

Thread name: MainThread.... inside generate_topics
['Tiger', 'Elephant', 'Lion', 'Zebra']


{'subjects': ['Tiger', 'Elephant', 'Lion', 'Zebra']}

In [23]:
import threading

class Joke(BaseModel):
    joke: str

def generate_joke(state: jokeState):
    
    thread_name = threading.current_thread().name
    print(f"Thread name: {thread_name}.... inside generate_joke")
    
    joke_prompt = """Generate a funny joke about the following subject: {subject}"""
    
    prompt = joke_prompt.format(subject=state["subject"])
    
    response = model.with_structured_output(Joke).invoke(prompt)
    
    return {"jokes":[response.joke]}

In [24]:
best_joke_prompt = """Below are a bunch of jokes about {topic}. Select the best one! Return ID of the best joke. {jokes}"""


def best_joke(state: OverallState):
    
    thread_name = threading.current_thread().name
    
    jokes = "\n\n".join(state["jokes"])
    
    prompt = best_joke_prompt.format(topic=state["topic"], jokes=jokes)
    response = model.invoke(prompt)
    return {"best_selected_joke":response.content}

In [25]:
from langgraph.constants import Send

def continue_to_joke(state: OverallState):
    thread_name = threading.current_thread().name
    print(f"Thread name: {thread_name}.... continue_to_joke")
    
    sends = [Send("generate_joke", {"subject": subject}) for subject in state["subjects"]]
    
    return sends

C:\Users\Aditya\AppData\Local\Temp\ipykernel_25072\609765306.py:1: LangGraphDeprecatedSinceV10: Importing Send from langgraph.constants is deprecated. Please use 'from langgraph.types import Send' instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  from langgraph.constants import Send


In [26]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(state_schema=OverallState)

graph.add_node("generate_topics", generate_topics)
graph.add_node("generate_joke", generate_joke)
graph.add_node("best_joke", best_joke)

graph.add_conditional_edges("generate_topics", continue_to_joke, ["generate_joke"])

graph.add_edge(START, "generate_topics")
graph.add_edge("generate_joke", "best_joke")
graph.add_edge("best_joke", END)

app = graph.compile()

# here we have not connetted generate_topics to generate_joke because we will be doing that dynamically and parallelly and that logic is in the function continue_to_joke

In [27]:
result = app.invoke({"topic":"Animal"})

print(result)

Thread name: MainThread.... inside generate_topics
['lion', 'elephant', 'tiger', 'bear']
Thread name: MainThread.... continue_to_joke
Thread name: ThreadPoolExecutor-2_0.... inside generate_joke
Thread name: ThreadPoolExecutor-2_1.... inside generate_joke
Thread name: ThreadPoolExecutor-2_2.... inside generate_joke
Thread name: ThreadPoolExecutor-2_3.... inside generate_joke
{'topic': 'Animal', 'subjects': ['lion', 'elephant', 'tiger', 'bear'], 'jokes': ["What do you call a lion that's wearing a coat? A shrouded lion.", 'Why did the elephant cross the road? To get to the other side!', 'What do you call a tiger that loves to go to the beach? A sand-tiger!', 'What do you call a bear with no teeth? A gummy bear!'], 'best_selected_joke': "Here's the ID of the best joke:\n\n**3**"}
